# 02 — Data preparation & EDA

**Input:**  `ctl_training_dev.sudsuay_titanic`
**Output:** `ctl_training_dev.sudsuay_titanic_prep` (silver / model-ready layer)

Three passes over the data:

1. **Profile** — nulls, constant columns, label integrity.
2. **EDA** — who survived, and along which axes.
3. **Prepare** — drop dead weight, impute, engineer features, stamp a reproducible
   train/test split, and write the result.

In [ ]:
from pyspark.sql import SparkSession, functions as F, Window
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

spark = SparkSession.builder.appName("titanic_prep_eda").getOrCreate()

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25,
                     "axes.spines.top": False, "axes.spines.right": False})

In [ ]:
CATALOG      = None
SCHEMA       = "ctl_training_dev"
SOURCE_TABLE = "sudsuay_titanic"
TARGET_TABLE = "sudsuay_titanic_prep"
TABLE_FMT    = "delta"

_p = lambda t: ".".join(x for x in [CATALOG, SCHEMA, t] if x)
SOURCE_FQN, TARGET_FQN = _p(SOURCE_TABLE), _p(TARGET_TABLE)

raw = spark.table(SOURCE_FQN).drop("_ingested_at").cache()
print(f"{SOURCE_FQN}: {raw.count():,} rows x {len(raw.columns)} columns")

## 1. Data quality profile

### 1.1 Nulls and cardinality

In [ ]:
n = raw.count()

profile = []
for c, t in raw.dtypes:
    stats = raw.select(
        F.sum(F.col(c).isNull().cast("int")).alias("nulls"),
        F.countDistinct(c).alias("distinct"),
        F.min(c).alias("min"),
        F.max(c).alias("max"),
    ).first()
    profile.append({
        "column": c, "type": t,
        "nulls": stats["nulls"], "null_%": round(100 * stats["nulls"] / n, 2),
        "distinct": stats["distinct"], "min": stats["min"], "max": stats["max"],
    })

profile_pdf = pd.DataFrame(profile)
profile_pdf

### 1.2 Constant columns

The Kaggle upload padded the file with 19 identical `zero` columns. They carry no signal
and nothing downstream should see them.

In [ ]:
constant_cols = profile_pdf.loc[profile_pdf["distinct"] <= 1, "column"].tolist()
print(f"{len(constant_cols)} zero-variance columns to drop:")
print(", ".join(constant_cols))

### 1.3 Label integrity — the important one

This file is Kaggle's `train.csv` and `test.csv` stacked together. The test half has no
published labels, so `Survived` was filled with `0` for every one of those rows. Training on
them would teach the model that 418 passengers died when we simply don't know.

In [ ]:
label_audit = (
    raw.withColumn("kaggle_half", F.when(F.col("PassengerId") <= 891, "train (labelled)")
                                   .otherwise("test (labels are filler)"))
       .groupBy("kaggle_half")
       .agg(F.count("*").alias("rows"),
            F.sum("Survived").alias("survived_1"),
            F.round(F.avg("Survived"), 3).alias("survival_rate"))
       .orderBy("kaggle_half")
)
label_audit.show(truncate=False)

In [ ]:
LABEL_CUTOFF = 891   # PassengerId <= 891 -> genuinely labelled

raw = raw.withColumn("is_labelled", (F.col("PassengerId") <= LABEL_CUTOFF).cast("boolean"))

print("Apparent survival rate on all rows :",
      round(raw.agg(F.avg("Survived")).first()[0], 3))
print("True survival rate on labelled rows:",
      round(raw.filter("is_labelled").agg(F.avg("Survived")).first()[0], 3))
print("\nEDA and model training below use labelled rows only.")

## 2. Exploratory data analysis

The labelled subset is 891 rows, so it moves to pandas for plotting. Categorical codes in
this file are numeric; the maps below match the original Kaggle encoding.

In [ ]:
pdf = raw.filter("is_labelled").drop(*constant_cols).toPandas()

SEX_MAP      = {0: "male", 1: "female"}
EMBARKED_MAP = {0.0: "Cherbourg", 1.0: "Queenstown", 2.0: "Southampton"}

pdf["sex_label"]      = pdf["Sex"].map(SEX_MAP)
pdf["embarked_label"] = pdf["Embarked"].map(EMBARKED_MAP).fillna("unknown")
pdf["class_label"]    = pdf["Pclass"].map({1: "1st", 2: "2nd", 3: "3rd"})

print(pdf.shape)
pdf.head()

### 2.1 Target balance

In [ ]:
counts = pdf["Survived"].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(4.5, 3.2))
bars = ax.bar(["died", "survived"], counts.values, color=["#B4413C", "#2E8B7A"], width=0.6)
ax.bar_label(bars, fmt="%d", padding=3)
ax.set_ylabel("passengers")
ax.set_title(f"Class balance — {counts[1] / counts.sum():.1%} survived")
ax.set_ylim(0, counts.max() * 1.15)
plt.tight_layout(); plt.show()

print(f"imbalance ratio: {counts[0] / counts[1]:.2f} : 1  (mild — no resampling needed, "
      f"but AUC/F1 beat accuracy as headline metrics)")

### 2.2 Survival by categorical feature

In [ ]:
cat_specs = [("sex_label", "Sex"), ("class_label", "Passenger class"),
             ("embarked_label", "Port of embarkation")]

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
for ax, (col, title) in zip(axes, cat_specs):
    g = pdf.groupby(col)["Survived"].agg(["mean", "size"]).sort_values("mean", ascending=False)
    bars = ax.bar(g.index.astype(str), g["mean"], color="#2E8B7A", width=0.6)
    ax.bar_label(bars, labels=[f"{v:.0%}\nn={int(s)}" for v, s in zip(g["mean"], g["size"])],
                 padding=3, fontsize=8)
    ax.axhline(pdf["Survived"].mean(), ls="--", lw=1, color="#B4413C")
    ax.set_ylim(0, 1.05); ax.set_ylabel("survival rate"); ax.set_title(title)
fig.suptitle("Survival rate by group (dashed line = overall rate)", y=1.04)
plt.tight_layout(); plt.show()

### 2.3 Continuous features

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 6.5))

for ax, col, bins in [(axes[0][0], "Age", 30), (axes[0][1], "Fare", 40)]:
    for val, lab, c in [(0, "died", "#B4413C"), (1, "survived", "#2E8B7A")]:
        ax.hist(pdf.loc[pdf.Survived == val, col], bins=bins, alpha=0.6, label=lab, color=c)
    ax.set_title(f"{col} distribution by outcome"); ax.set_xlabel(col); ax.legend()

for ax, col in [(axes[1][0], "Age"), (axes[1][1], "Fare")]:
    data = [pdf.loc[pdf.Survived == v, col] for v in (0, 1)]
    bp = ax.boxplot(data, patch_artist=True, widths=0.5)
    for patch, c in zip(bp["boxes"], ["#B4413C", "#2E8B7A"]):
        patch.set_facecolor(c); patch.set_alpha(0.65)
    ax.set_xticks([1, 2], ["died", "survived"])   # set after: boxplot's own kwarg was renamed in mpl 3.9
    ax.set_ylabel(col); ax.set_title(f"{col} spread by outcome")

axes[1][1].set_yscale("symlog")
plt.tight_layout(); plt.show()

print(pdf.groupby("Survived")[["Age", "Fare", "SibSp", "Parch"]].mean().round(2).to_string())

### 2.4 Family size

`SibSp` (siblings/spouses) and `Parch` (parents/children) are more informative together
than apart — travelling alone and travelling in a large group are both bad news.

In [ ]:
pdf["family_size"] = pdf["SibSp"] + pdf["Parch"] + 1

g = pdf.groupby("family_size")["Survived"].agg(["mean", "size"])
g = g[g["size"] >= 5]

fig, ax = plt.subplots(figsize=(6.5, 3.4))
bars = ax.bar(g.index.astype(str), g["mean"], color="#2E8B7A", width=0.65)
ax.bar_label(bars, labels=[f"n={int(s)}" for s in g["size"]], padding=3, fontsize=8)
ax.axhline(pdf["Survived"].mean(), ls="--", lw=1, color="#B4413C")
ax.set_xlabel("family size (self + SibSp + Parch)"); ax.set_ylabel("survival rate")
ax.set_title("Survival peaks for families of 2–4"); ax.set_ylim(0, 1.0)
plt.tight_layout(); plt.show()

### 2.5 Correlation

In [ ]:
num_cols = ["Survived", "Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked",
            "family_size"]
corr = pdf[num_cols].corr()

fig, ax = plt.subplots(figsize=(6.8, 5.6))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(num_cols)), num_cols, rotation=45, ha="right")
ax.set_yticks(range(len(num_cols)), num_cols)
for i in range(len(num_cols)):
    for j in range(len(num_cols)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=7.5,
                color="white" if abs(corr.iloc[i, j]) > 0.55 else "black")
ax.grid(False)
fig.colorbar(im, shrink=0.8)
ax.set_title("Pearson correlation (labelled rows)")
plt.tight_layout(); plt.show()

print(corr["Survived"].drop("Survived").sort_values(key=abs, ascending=False).round(3).to_string())

### EDA takeaways

- **Sex dominates.** Women survived at roughly four times the rate of men; it is by far the
  strongest single correlate.
- **Class and fare follow**, and they say much the same thing — 1st class and expensive
  tickets both track survival, so expect them to be partly redundant.
- **Age is weakly signalled on its own** but matters at the extremes (young children did
  better), which argues for a banded version alongside the raw value.
- **Family size is non-monotonic** — solo travellers and large families both fared badly,
  best around 2–4. Worth an explicit feature rather than leaving `SibSp`/`Parch` split.
- **Fare is heavily right-skewed** with 17 zero-fare tickets; a log transform tames it.
- **Embarked has 2 nulls** and is otherwise ~70% Southampton.

## 3. Data preparation

Every step here is a decision, so each one gets a comment saying why.

In [ ]:
prep = raw.drop(*constant_cols)          # 1. drop the 19 padding columns — no variance

# 2. Impute Embarked with the mode. Two rows; anything fancier is overfitting the gap.
embarked_mode = (prep.filter(F.col("Embarked").isNotNull())
                     .groupBy("Embarked").count()
                     .orderBy(F.desc("count")).first()["Embarked"])
prep = prep.withColumn("Embarked", F.coalesce("Embarked", F.lit(embarked_mode)))
print(f"Embarked nulls imputed with mode = {embarked_mode} (Southampton)")

# 3. Fare of 0 almost certainly means "unknown", not "free". Replace with the class median.
class_med = {r["Pclass"]: r["med"] for r in
             prep.filter(F.col("Fare") > 0).groupBy("Pclass")
                 .agg(F.expr("percentile_approx(Fare, 0.5)").alias("med")).collect()}
med_expr = F.create_map([x for k, v in class_med.items() for x in (F.lit(k), F.lit(float(v)))])
prep = prep.withColumn(
    "Fare_imputed", (F.col("Fare") == 0).cast("boolean")
).withColumn(
    "Fare", F.when(F.col("Fare") > 0, F.col("Fare")).otherwise(med_expr[F.col("Pclass")])
)
print("Fare medians by class:", {k: round(v, 2) for k, v in sorted(class_med.items())})

In [ ]:
# 4. Feature engineering
prep = (
    prep
    # household size and the two regimes that behave differently
    .withColumn("FamilySize", F.col("SibSp") + F.col("Parch") + F.lit(1))
    .withColumn("IsAlone", (F.col("SibSp") + F.col("Parch") == 0).cast("int"))
    .withColumn("SmallFamily", F.col("FamilySize").between(2, 4).cast("int"))
    # fare: log-scale to tame the skew, and per-head because Fare is priced per ticket
    .withColumn("LogFare", F.log1p("Fare"))
    .withColumn("FarePerPerson", F.round(F.col("Fare") / F.col("FamilySize"), 4))
    # age: keep the continuous value, add a banded view for the non-linear child effect
    .withColumn("IsChild", (F.col("Age") < 16).cast("int"))
    .withColumn("AgeBand", F.when(F.col("Age") < 12, 0)
                            .when(F.col("Age") < 18, 1)
                            .when(F.col("Age") < 30, 2)
                            .when(F.col("Age") < 45, 3)
                            .when(F.col("Age") < 60, 4)
                            .otherwise(5))
    # interaction the EDA points at directly: class and sex together
    .withColumn("ClassSex", F.col("Pclass") * F.lit(2) + F.col("Sex"))
)

# fare quartile bands, computed from the data rather than hard-coded
q1, q2, q3 = prep.approxQuantile("Fare", [0.25, 0.5, 0.75], 0.01)
prep = prep.withColumn("FareBand", F.when(F.col("Fare") <= q1, 0)
                                    .when(F.col("Fare") <= q2, 1)
                                    .when(F.col("Fare") <= q3, 2)
                                    .otherwise(3))
print(f"Fare quartile cuts: {q1:.2f} / {q2:.2f} / {q3:.2f}")

In [ ]:
# 5. Deterministic 80/20 split, written into the table so notebook 03 is reproducible
#    and so re-running this notebook never reshuffles which rows are held out.
SEED = 42
prep = prep.withColumn(
    "data_split",
    F.when(~F.col("is_labelled"), F.lit("holdout"))            # unlabelled Kaggle test half
     .when(F.abs(F.hash(F.concat_ws("_", F.col("PassengerId"), F.lit(SEED)))) % 100 < 80,
           F.lit("train"))
     .otherwise(F.lit("test"))
)

(prep.groupBy("data_split")
     .agg(F.count("*").alias("rows"),
          F.round(F.avg(F.when(F.col("is_labelled"), F.col("Survived"))), 3).alias("survival_rate"))
     .orderBy("data_split")
     .show(truncate=False))

In [ ]:
# 6. Final column order + types
FEATURE_COLS = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked",
                "FamilySize", "IsAlone", "SmallFamily", "LogFare", "FarePerPerson",
                "IsChild", "AgeBand", "FareBand", "ClassSex"]

prep_final = (
    prep.select(
        F.col("PassengerId").cast("int"),
        F.col("Survived").cast("int").alias("label"),
        *[F.col(c).cast("double") for c in FEATURE_COLS],
        F.col("Fare_imputed"),
        F.col("is_labelled"),
        F.col("data_split"),
    )
    .withColumn("_prepared_at", F.current_timestamp())
)

prep_final.printSchema()
print(f"{prep_final.count():,} rows x {len(prep_final.columns)} columns")
prep_final.limit(5).toPandas()

In [ ]:
# 7. Post-prep assertions — cheap insurance against a silently broken table
assert prep_final.filter(F.col("label").isNull()).count() == 0, "null labels"
nulls_in_features = {c: prep_final.filter(F.col(c).isNull()).count() for c in FEATURE_COLS}
bad = {k: v for k, v in nulls_in_features.items() if v}
assert not bad, f"nulls remain in features: {bad}"
assert prep_final.select("PassengerId").distinct().count() == prep_final.count(), "dup ids"
print("all checks passed — no nulls in features, labels complete, ids unique")

## 4. Write to `ctl_training_dev.sudsuay_titanic_prep`

In [ ]:
schema_fqn = ".".join(x for x in [CATALOG, SCHEMA] if x)
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema_fqn}")


def save_table(sdf, fqn, fmt="delta"):
    try:
        (sdf.write.format(fmt).mode("overwrite")
            .option("overwriteSchema", "true").saveAsTable(fqn))
        print(f"wrote {fqn} as {fmt}")
    except Exception as exc:
        reason = str(exc).strip().splitlines()[0][:160]
        print(f"[warn] {fmt} write failed ({type(exc).__name__}: {reason}); retrying as parquet")
        sdf.write.format("parquet").mode("overwrite").saveAsTable(fqn)
        print(f"wrote {fqn} as parquet")


save_table(prep_final, TARGET_FQN, TABLE_FMT)

In [ ]:
check = spark.table(TARGET_FQN)
print(f"{TARGET_FQN}: {check.count():,} rows x {len(check.columns)} columns")
check.groupBy("data_split").count().orderBy("data_split").show()

---

**Done.** `ctl_training_dev.sudsuay_titanic_prep` is model-ready: 16 features, a clean
`label`, an `is_labelled` flag that quarantines the unlabelled Kaggle half, and a frozen
`data_split` column.

Next: **03 — train & evaluate** with MLflow tracking.